# Comparación de modelos preentrenados para clasificación de sentimiento en español




In [1]:
!pip -q install "transformers>=4.44,<5" "torch>=2.2" "pandas>=2.0" "scikit-learn>=1.3" "huggingface_hub>=0.25" "accelerate>=0.30" "matplotlib>=3.7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os, time, json, platform, getpass
import numpy as np
import pandas as pd
import torch
from transformers import pipeline, AutoConfig
from huggingface_hub import login
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print('PyTorch:',torch.__version__)
# Para modelos públicos, la autenticación es opcional.
if os.getenv('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
else:
    print('Sin HF_TOKEN: acceso anónimo a modelos públicos.')

GPU disponible: True
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Sin HF_TOKEN: acceso anónimo a modelos públicos.


## 2. Dataset


In [3]:
from pathlib import Path
path=Path('data/mensajes.csv')
if not path.exists():
    try:
        from google.colab import files
        uploaded=files.upload()
        path=Path('mensajes.csv')
    except ImportError:
        raise FileNotFoundError('Coloque data/mensajes.csv en el directorio de trabajo.')
df=pd.read_csv(path,encoding='utf-8-sig')
assert {'id','texto','etiqueta','origen','idioma'} <= set(df.columns)
assert df.etiqueta.isin(['positivo','neutral','negativo']).all()
assert df.id.is_unique
print(df.etiqueta.value_counts())
display(df.head())

Saving mensajes.csv to mensajes.csv
etiqueta
positivo    12
neutral     12
negativo    12
Name: count, dtype: int64


,id,texto,etiqueta,origen,idioma
0,M010,Recomiendo este servicio por su calidad.,positivo,sintetico_manual,es
1,M013,Quiero conocer el horario de atención.,neutral,sintetico_manual,es
2,M006,La transferencia se realizó sin problemas.,positivo,sintetico_manual,es
3,M028,El servicio fue lento y poco claro.,negativo,sintetico_manual,es
4,M033,Me cobraron una comisión que no esperaba.,negativo,sintetico_manual,es


## 3. Modelos y normalización
Los modelos emplean distintas etiquetas nativas; se homologan antes de evaluar. Para el modelo de estrellas se utiliza 1–2 negativo, 3 neutral y 4–5 positivo.

In [4]:
MODELS = [
 'cardiffnlp/twitter-xlm-roberta-base-sentiment',
 'lxyuan/distilbert-base-multilingual-cased-sentiments-student',
 'nlptown/bert-base-multilingual-uncased-sentiment',
]
def normalizar(label, model):
    s=str(label).strip().lower()
    if 'nlptown/' in model:
        stars=int(s[0]); return 'negativo' if stars<=2 else ('neutral' if stars==3 else 'positivo')
    aliases={'negative':'negativo','neutral':'neutral','positive':'positivo',
             'negativo':'negativo','positivo':'positivo','label_0':'negativo',
             'label_1':'neutral','label_2':'positivo'}
    if s not in aliases: raise ValueError(f'Etiqueta inesperada: {label} para {model}')
    return aliases[s]
print('Modelos a evaluar:',len(MODELS))

Modelos a evaluar: 3


## 4. Inferencia, métricas y latencia
Se ejecuta inferencia secuencial con batch_size=1 para obtener una medición comparable de latencia por mensaje. Se registra el tiempo de carga aparte. La primera inferencia se usa como calentamiento. La latencia varía con el hardware y no equivale a un benchmark de producción.

In [5]:
from pathlib import Path
Path('results').mkdir(exist_ok=True)
records=[]; predictions=[]; errors=[]
for model_id in MODELS:
    print('Evaluando:',model_id)
    try:
        t0=time.perf_counter()
        clf=pipeline('text-classification',model=model_id,tokenizer=model_id,
                     device=0 if torch.cuda.is_available() else -1)
        load_s=time.perf_counter()-t0
        _=clf('Mensaje de prueba.',truncation=True)
        y_pred=[]; lat=[]
        for _,r in df.iterrows():
            t0=time.perf_counter()
            out=clf(r.texto,truncation=True,max_length=256)[0]
            if torch.cuda.is_available(): torch.cuda.synchronize()
            lat.append((time.perf_counter()-t0)*1000)
            y_pred.append(normalizar(out['label'],model_id))
            predictions.append({'id':r.id,'modelo':model_id,'real':r.etiqueta,'prediccion':y_pred[-1], 'latencia_ms':lat[-1]})
        p,r,f,_=precision_recall_fscore_support(df.etiqueta,y_pred,average='macro',zero_division=0)
        records.append({'modelo':model_id,'accuracy':accuracy_score(df.etiqueta,y_pred),
          'precision_macro':p,'recall_macro':r,'f1_macro':f,
          'latencia_media_ms':float(np.mean(lat)),'latencia_p95_ms':float(np.percentile(lat,95)),
          'carga_s':load_s,'n':len(df),'dispositivo':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
        print(classification_report(df.etiqueta,y_pred,zero_division=0))
        del clf
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        errors.append({'modelo':model_id,'error':repr(e)})
        print('ERROR:',repr(e))
res=pd.DataFrame(records); display(res)
if not res.empty: res.to_csv('results/metricas.csv',index=False)
pd.DataFrame(predictions).to_csv('results/predicciones.csv',index=False)
pd.DataFrame(errors,columns=['modelo','error']).to_csv('results/errores.csv',index=False)
assert len(records)==len(MODELS), 'Uno o más modelos fallaron: revisar results/errores.csv'

Evaluando: cardiffnlp/twitter-xlm-roberta-base-sentiment


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


              precision    recall  f1-score   support

    negativo       0.86      1.00      0.92        12
     neutral       1.00      0.33      0.50        12
    positivo       0.67      1.00      0.80        12

    accuracy                           0.78        36
   macro avg       0.84      0.78      0.74        36
weighted avg       0.84      0.78      0.74        36



,modelo,accuracy,precision_macro,recall_macro,f1_macro,latencia_media_ms,latencia_p95_ms,carga_s,n,dispositivo
0,cardiffnlp/twitter-xlm-roberta-base-sentiment,0.972222,0.974359,0.972222,0.972174,11.643426,15.383945,26.132527,36,Tesla T4
1,lxyuan/distilbert-base-multilingual-cased-sent...,0.472222,0.404187,0.472222,0.412698,4.829484,5.365678,10.651309,36,Tesla T4
2,nlptown/bert-base-multilingual-uncased-sentiment,0.777778,0.841270,0.777778,0.741026,8.905761,9.668402,61.754059,36,Tesla T4


In [8]:
from google.colab import files

# Guardar los resultados de los tres modelos (usando la variable correcta 'res')
res.to_csv("metricas_modelos.csv", index=False)

# Descargar el archivo a la computadora
files.download("metricas_modelos.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Comparación y conclusión parcial
Se recomienda priorizar F1 macro por la importancia equivalente de las tres clases. La regla de decisión es ilustrativa: entre los modelos con F1 macro a menos de 0,02 del máximo, se selecciona el de menor latencia media. Debe revisarse también el recall de la clase negativa y validarse con datos reales antes de adoptar el modelo.

In [6]:
best_f1=res.f1_macro.max()
candidates=res[res.f1_macro >= best_f1-0.02]
recommended=candidates.sort_values(['latencia_media_ms','f1_macro'],ascending=[True,False]).iloc[0]
print('Recomendación provisional:',recommended.modelo)
print('F1 macro:',round(recommended.f1_macro,4),'Latencia media (ms):',round(recommended.latencia_media_ms,2))
print('Conclusión: válida únicamente para el conjunto sintético de evaluación.')
with open('results/recomendacion.json','w',encoding='utf-8') as f:
 json.dump({'modelo':recommended.modelo,'regla':'F1 macro >= max - 0.02; menor latencia'},f,ensure_ascii=False,indent=2)

Recomendación provisional: cardiffnlp/twitter-xlm-roberta-base-sentiment
F1 macro: 0.9722 Latencia media (ms): 11.64
Conclusión: válida únicamente para el conjunto sintético de evaluación.
